# 📏 Sesiunea 6 · Auditul

Un muzeu a moștenit două sisteme de la un colaborator plecat. Fără documentație.

| | ce face | cum |
|---|---|---|
| **A** | estimează prețul biletului | regresie liniară |
| **B** | semnalează șansa de supraviețuire | `if Fare > 50` |

**Sarcina ta:** decide dacă pot fi folosite.

Regula zilei: la fiecare 🔮 **ghicești înainte să rulezi**. La fiecare 🛠️ **scrii tu codul**.


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_curve, roc_auc_score)
plt.rcParams["figure.figsize"] = (8, 4.5)

df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
df["Age"] = df["Age"].fillna(df["Age"].median())
df["SexNum"] = (df["Sex"] == "male").astype(int)
train, test = train_test_split(df, test_size=0.2, random_state=0)

print(len(train), "train /", len(test), "test")

❓ De ce nu evaluăm pe datele de antrenare, unde am avea mai multe rânduri?

<details><summary>💡</summary>
Un model poate memora. Acolo ar părea excelent și ar eșua pe date noi. Regula din industrie: dacă te-ai uitat la test ca să iei o decizie, nu mai e test.
</details>

---
# A · Estimatorul de preț

In [ ]:
FEATURES = ["Pclass", "Age", "SibSp", "Parch", "SexNum"]
sistem_A = LinearRegression().fit(train[FEATURES], train["Fare"])

pred = sistem_A.predict(test[FEATURES])
real = test["Fare"].values
rez  = real - pred

plt.scatter(pred, rez, s=18, alpha=0.5); plt.axhline(0, c="k", lw=1.5)
plt.xlabel("prezis"); plt.ylabel("rezidual (real − prezis)"); plt.grid(alpha=0.3); plt.show()

❓ Norul are o formă. Care, și ce înseamnă pentru raport?

❓ Ce ar însemna dacă punctele ar forma o curbă, nu o pâlnie?

<details><summary>💡</summary>
Pâlnie: eroarea crește cu prețul, deci o singură cifră nu descrie corect tot intervalul. Pe bilete scumpe sistemul e mult mai prost decât media.<br><br>
Curbă: relația nu e liniară, iar modelul ratează sistematic. Se repară cu variabile polinomiale (sesiunea 4), nu cu mai multe date.
</details>

## 🛠️ 1 · Cele trei metrici, de la zero

🔮 Înainte de a rula: ce va da `rez.mean()`?

In [ ]:
mae  = # ...  np.abs(rez).mean()
mse  = # ...  (rez ** 2).mean()
rmse = # ...  np.sqrt(mse)

print(f"media brută: {rez.mean():.2f}")
print(f"MAE {mae:.2f} | MSE {mse:.2f} | RMSE {rmse:.2f}")
assert np.allclose([mae, mse, rmse],
                   [mean_absolute_error(real, pred), mean_squared_error(real, pred),
                    mean_squared_error(real, pred) ** 0.5]), "nu se potrivește cu sklearn"
print("✓")

❓ De ce media brută e ~0 deși modelul greșește constant?

❓ RMSE / MAE = 1.71. Ce citești din raportul ăsta?

<details><summary>💡</summary>
Greșelile pozitive și negative se anulează. De asta luăm valoarea absolută sau pătratul.<br><br>
Raport mare = câteva greșeli uriașe trag RMSE-ul sus. Raport aproape de 1 = greșeli uniforme. E un diagnostic gratuit, folosibil la orice model de regresie.
</details>

## 🛠️ 2 · Cât de fragil e RMSE?

Găsește **câte** predicții trebuie scoase (cele mai proaste) ca RMSE să coboare sub 25.

🔮 Ghicește numărul înainte.

In [ ]:
ordine = np.argsort(np.abs(rez))            # crescător după mărimea greșelii

for k in range(0, 20):
    pastrate = ordine[:len(ordine) - k] if k else ordine
    r = # ...  mean_squared_error(real[pastrate], pred[pastrate]) ** 0.5
    m = mean_absolute_error(real[pastrate], pred[pastrate])
    print(f"scoatem {k:>2} → RMSE {r:6.2f} | MAE {m:6.2f}")
    if r < 25:
        print(f"\n{k} pasageri din {len(real)} = {100*k/len(real):.1f}% din date")
        break

❓ Ce procent din date ținea RMSE-ul sus?

❓ Ai scoate acei pasageri din dataset? De ce da, de ce nu?

<details><summary>💡</summary>
Sub 3%.<br><br>
Nu automat. Un outlier poate fi o eroare de măsurare (se scoate) sau un caz real și important (nu se scoate). Biletele de sute de lire chiar au existat. Dacă le scoți, modelul nu mai știe nimic despre clasa I, iar muzeul exact acolo are cele mai multe întrebări.
</details>

## 🛠️ 3 · R², de la zero

MAE e în lire, deci nu spune dacă e bine. Față de ce?

R² = 1 − (suma pătratelor erorilor modelului) / (suma pătratelor erorilor modelului „media”)

In [ ]:
media = train["Fare"].mean()

ss_model = # ...  ((real - pred) ** 2).sum()
ss_media = # ...  ((real - real.mean()) ** 2).sum()
r2 = 1 - ss_model / ss_media

print(f"R² manual  = {r2:.3f}")
print(f"R² sklearn = {r2_score(real, pred):.3f}")
print(f"R² al modelului 'media' = {r2_score(real, np.full(len(real), media)):.3f}")

❓ De ce R²-ul modelului „media” e fix 0?

❓ Poate fi R² negativ? Ce ar însemna?

❓ Un coleg raportează R² = 0.46 și zice „modelul e corect în 46% din cazuri”. Ce e greșit?

❓ La numitor am folosit media pe **test**. Ce s-ar schimba dacă am folosi media pe train?

<details><summary>💡</summary>
Pentru că R² e <i>definit</i> ca îmbunătățirea față de acel model; el se îmbunătățește pe sine cu zero.<br><br>
Da. Negativ = mai prost decât media. Apare tipic la overfitting evaluat pe test.<br><br>
R² nu e procent de corectitudine. La regresie nu există „corect”, doar „aproape”. E fracțiunea din variație explicată în plus față de media simplă.<br><br>
Rezultatul s-ar schimba puțin (0.463 în loc de 0.462). `sklearn` folosește media setului pe care evaluezi. Diferența e mică aici, dar pe seturi mici sau cu distribuții diferite între train și test poate deveni vizibilă. Merită știut ce anume calculează funcția pe care o chemi.
</details>

## ✅ Verdict A

In [ ]:
verdict_A = {
    "utilizabil": # ...  "doar orientativ",
    "de ce": # ...  "R² 0.46 și eroarea crește cu prețul",
}
verdict_A

---
# B · Alerta de supraviețuire

🔮 Ce acuratețe are un singur `if`?

In [ ]:
y_true   = test["Survived"].values
sistem_B = (test["Fare"] > 50).astype(int).values

print(f"acuratețe sistem B: {accuracy_score(y_true, sistem_B):.3f}")

## 🛠️ 4 · Cel mai bun sistem posibil care nu știe nimic

Construiește predicția care maximizează acuratețea **fără să se uite la niciun pasager**.

🔮 Ce acuratețe va lua?

In [ ]:
clasa_majoritara = # ...  int(train["Survived"].mode()[0])
nimic = np.full(len(y_true), clasa_majoritara)

print(f"sistem B : {accuracy_score(y_true, sistem_B):.3f}")
print(f"nimic    : {accuracy_score(y_true, nimic):.3f}")
print(f"câștig real: {accuracy_score(y_true, sistem_B) - accuracy_score(y_true, nimic):+.3f}")

❓ Ce s-a întâmplat cu cei 73%?

❓ Ce idee din partea A e exact aceeași?

<details><summary>💡</summary>
Erau mari doar pentru că nu aveau adversar. Câștigul real e 11 puncte, nu 73.<br><br>
R². Acolo adversarul era media, aici e clasa majoritară. Aceeași idee: <b>o cifră nu înseamnă nimic fără un baseline</b>.
</details>

## 🛠️ 5 · Matricea de confuzie, fără sklearn

🔮 Ghicește care dintre FP și FN va fi mai mare.

In [ ]:
tp = # ...  int(((sistem_B == 1) & (y_true == 1)).sum())
fp = # ...  int(((sistem_B == 1) & (y_true == 0)).sum())
fn = # ...  int(((sistem_B == 0) & (y_true == 1)).sum())
tn = # ...  int(((sistem_B == 0) & (y_true == 0)).sum())

print(f"TN {tn:>4}   FP {fp:>4}\nFN {fn:>4}   TP {tp:>4}")
assert (np.array([[tn, fp], [fn, tp]]) == confusion_matrix(y_true, sistem_B)).all()
print("\n✓")

❓ 6 alarme false, 42 de ratări. De ce atât de dezechilibrat?

❓ Ce ar trebui schimbat în regulă ca raportul să se inverseze?

<details><summary>💡</summary>
Pragul de 50 e sever: sistemul spune DA rar, deci greșește rar când o face, dar pierde toți supraviețuitorii cu bilete ieftine.<br><br>
Coborârea pragului. O regulă strictă cumpără siguranță cu prețul acoperirii, și invers.
</details>

## 🛠️ 6 · Precision, recall, F1, de la zero

🔮 Care dintre precision și recall va fi mare?

In [ ]:
precizie  = # ...  tp / (tp + fp)
acoperire = # ...  tp / (tp + fn)
f1        = # ...  2 * precizie * acoperire / (precizie + acoperire)

print(f"precision {precizie:.3f} | recall {acoperire:.3f} | F1 {f1:.3f}")
print(f"media obișnuită a primelor două: {(precizie + acoperire) / 2:.3f}")
assert np.allclose([precizie, acoperire, f1],
                   [precision_score(y_true, sistem_B), recall_score(y_true, sistem_B),
                    f1_score(y_true, sistem_B)])
print("✓")

❓ Precision citește o coloană din matrice, recall citește un rând. Care pe care?

❓ Muzeul vrea un panou care afirmă *„un pasager ca tine ar fi supraviețuit”*. Precision sau recall?

❓ Dar un screening medical?

<details><summary>💡</summary>
Precision → coloana „Prezis DA”. Recall → rândul „Adevărat DA”. Amândouă au TP la numărător.<br><br>
Panou: precision. O afirmație greșită e jenantă public, o omisiune nu supără pe nimeni.<br><br>
Screening: recall. O ratare costă o viață, o alarmă falsă costă un set de analize. <b>Alegerea metricii e o decizie de produs, nu una tehnică.</b>
</details>

## 🛠️ 7 · Sparge F1

Construiește un vector de predicții cu **precision = 1.0** și **F1 sub 0.05**.

🔮 Câți de 1 crezi că îți trebuie?

In [ ]:
un_singur_da = np.zeros(len(y_true), dtype=int)
i = # ...  np.where(y_true == 1)[0][0]
un_singur_da[i] = 1

p = precision_score(y_true, un_singur_da)
r = recall_score(y_true, un_singur_da)
print(f"precision {p:.3f} | recall {r:.3f}")
print(f"media obișnuită {(p + r) / 2:.3f}  vs  F1 {f1_score(y_true, un_singur_da):.3f}")

❓ Media obișnuită dă 0.5, F1 dă ~0.03. Care e verdictul onest?

❓ De ce F1 folosește media armonică?

<details><summary>💡</summary>
F1. Un sistem care spune DA o singură dată e inutil, oricât de „precis” ar fi.<br><br>
Media armonică e trasă în jos de valoarea mică. Nu poți compensa o acoperire catastrofală cu o precizie perfectă. Asta e chiar scopul ei.
</details>

## 🛠️ 8 · Când acuratețea devine inutilizabilă

Găsește proporția de pozitive sub care sistemul „nimic” trece de 95% acuratețe.

In [ ]:
for procent in [50, 40, 30, 20, 10, 5, 2, 1]:
    n = 5000
    y = np.zeros(n, dtype=int)
    y[:int(n * procent / 100)] = 1
    acc = # ...  accuracy_score(y, np.zeros(n, dtype=int))
    flag = "  ← inutil, dar 'excelent'" if acc > 0.95 else ""
    print(f"{procent:>3}% pozitive → acuratețe {acc:.3f}{flag}")

❓ La ce prag de dezechilibru devine acuratețea o minciună?

❓ Ce metrică ai raporta în locul ei, pentru fraudă bancară?

<details><summary>💡</summary>
Sub ~5% pozitive, dar de fapt problema începe mult mai devreme. Titanic e la 38% și acuratețea deja induce în eroare.<br><br>
Recall pe clasa rară, plus precision, plus matricea întreagă. Prima întrebare la orice clasificare: <b>cât de dezechilibrate sunt clasele?</b>
</details>

## ✅ Verdict B

In [ ]:
verdict_B = {
    "utilizabil": # ...  "nu la pragul actual",
    "de ce": # ...  "recall 0.39, ratează 6 din 10",
}
verdict_B